In [1]:
## DataStax DB

In [11]:
!pip install \
    "langchain>=0.3.23,<0.4" \
    "langchain-core>=0.3.52,<0.4" \
    "langchain-astradb>=0.6,<0.7"

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
from pathlib import Path


def load_project_dotenv() -> None:
    """Find repo-root .env (notebooks often run with cwd in a subfolder)."""
    here = Path.cwd().resolve()
    for directory in [here, *here.parents]:
        candidate = directory / ".env"
        if candidate.is_file():
            load_dotenv(candidate, override=True)
            return
    load_dotenv()


def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or not str(value).strip():
        raise RuntimeError(
            f"Missing {name}. Set it in the project root .env, then re-run this cell."
        )
    return value


load_project_dotenv()

os.environ["OPENAI_API_KEY"] = require_env("OPENAI_API_KEY")
os.environ["ASTRA_DB_API_ENDPOINT"] = require_env("ASTRA_DB_API_ENDPOINT")
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = require_env("ASTRA_DB_APPLICATION_TOKEN")

In [15]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small', dimensions=1024)

In [16]:
import os

from langchain_astradb import AstraDBVectorStore

vector_store = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=os.environ["ASTRA_DB_API_ENDPOINT"],
    collection_name="astra_vector_langchain",
    token=os.environ["ASTRA_DB_APPLICATION_TOKEN"],
    namespace=None,
)

vector_store

In [17]:
## Create Documents
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [18]:
vector_store.add_documents(documents = documents)

['d8c98c4e35d641a6ad31b06d0c2827d2',
 '59db6c52b434400f9ed47f331eadd506',
 '62286c956d12417cb1a4a8cbd2331378',
 'b46e5c2305f140f7937c7c996a6cefc5',
 '1d53960b61d44a2c896132d268f51bde',
 'ac7df05b00cd48acadd2261e6baa626c',
 '6d28ca66fa02498ca347be43fd85c1d9',
 'b7d49efdec0846c78632cdd5b2b2f94b',
 '93bcbcb5039b41f1aa1dc91c75968573',
 'aca92a3562ca47479e3d0652a48ac8a8']

In [19]:
### Search from Vector Store DB

vector_store.similarity_search("What is the weather")

[Document(id='59db6c52b434400f9ed47f331eadd506', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='93bcbcb5039b41f1aa1dc91c75968573', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='62286c956d12417cb1a4a8cbd2331378', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='aca92a3562ca47479e3d0652a48ac8a8', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [20]:
results  = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k = 3,
    filter  = {"source": "tweet"},
)

for res in results:
  print(f'* "{res.page_content}", metadata = {res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* "Wow! That was an amazing movie. I can't wait to see it again.", metadata = {'source': 'tweet'}


In [21]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstraction to make working with LLMs easy",
    k = 3,
    filter  = {"source": "tweet"}
)

for res, score in results:
  print(f'* [SIM={score:.2f}]"{res.page_content}",metadata  = {res.metadata}')

* [SIM=0.72]"Building an exciting new project with LangChain - come check it out!",metadata  = {'source': 'tweet'}
* [SIM=0.71]"LangGraph is the best framework for building stateful, agentic applications!",metadata  = {'source': 'tweet'}
* [SIM=0.53]"Wow! That was an amazing movie. I can't wait to see it again.",metadata  = {'source': 'tweet'}


In [22]:
### Retriever
retriever=vector_store.as_retriever(
  search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='b46e5c2305f140f7937c7c996a6cefc5', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [23]:
### Retriever
retriever=vector_store.as_retriever(
  search_type="mmr",
    search_kwargs={"k": 1},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='b46e5c2305f140f7937c7c996a6cefc5', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]